# APT Lab 01 — GRPO Ratio Clipping and Gradient Diagnostics

**Learning path:** toy GRPO clipping → mathematical diagnostic → tested observation → reuse in TRL GRPO.

**Status:** guided student lab, **not** a real GRPO trainer or an executed LLM training result. Python/PyTorch and Matplotlib on CPU are sufficient.

**Learning goals:** understand GRPO/PPO-style importance ratios, advantages, upper vs lower clipping branches, the difference between ratio clipping and gradient norm clipping, automatic differentiation, gradient cosine, and diagnostic evidence. **Four visualization experiments** below make these mechanisms inspectable.

**Workflow:** predict → complete TODOs → run → interpret plots → test → change parameters → record the result. In Colab select **Runtime → Run all** after completing TODOs. If saving edits, use **File → Save a copy in Drive**; editing Colab does not automatically commit your GitHub repository.

## 0. Setup and reproducibility

Colab normally has PyTorch installed. This cell verifies it rather than silently upgrading the runtime.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
print('PyTorch:', torch.__version__)
print('Device: CPU (sufficient for this lab)')
torch.manual_seed(7)

## 1. Predict before coding

For the two samples below, the initial probability ratios are `1.3` and `1.0`. Both have advantage `+1`. With `epsilon = 0.2`, which sample should be suppressed by GRPO ratio clipping? Write your prediction here before running the next cell.

> **My prediction:** ...

Recall that clipping the **importance ratio in the loss** is not the same as clipping the **gradient norm**.

## 2. Build a differentiable toy ratio model

We use `log_ratio = log(initial_ratio) + feature · theta`. The feature vectors are an explicit local model of how log probabilities change with parameters. They are **not** a complete normalized token policy, so this is a controlled mechanism demonstration.

In [ ]:
theta = torch.nn.Parameter(torch.zeros(2))
features = torch.tensor([[2.0, 0.0], [0.0, 1.0]])
initial_ratios = torch.tensor([1.3, 1.0])
advantages = torch.tensor([1.0, 1.0])
epsilon = 0.2

log_ratios = torch.log(initial_ratios) + features @ theta
ratios = torch.exp(log_ratios)
print('Ratios:', ratios.detach().tolist())

## 3. Your implementation — two lines

The positive-advantage clipped objective is `min(ratio * advantage, clamp(ratio, 1-epsilon, 1+epsilon) * advantage)`.

Replace the two `...` values. **Don't reveal the reference solution until after attempting this.**

In [ ]:
# TODO 1: multiply each probability ratio by its advantage.
unclipped = ...
# TODO 2: clamp each ratio to [1-epsilon, 1+epsilon], then multiply by its advantage.
clipped = ...

assert isinstance(unclipped, torch.Tensor), 'Implement unclipped first'
assert isinstance(clipped, torch.Tensor), 'Implement clipped first'
assert unclipped.shape == (2,) and clipped.shape == (2,)
print('Unclipped:', unclipped.detach().tolist())
print('Clipped:', clipped.detach().tolist())

## 4. Measure gradients and calculate cosine similarity

For each objective, `autograd.grad` returns the gradient with respect to `theta`. We compare the raw and clipped atomic gradients against a **fixed illustrative composition gradient** `[1, -1]`. A positive cosine means local alignment; negative means local conflict.

In [ ]:
raw_objective = unclipped.sum()
clipped_objective = torch.minimum(unclipped, clipped).sum()
g_raw = torch.autograd.grad(raw_objective, theta, retain_graph=True)[0]
g_clipped = torch.autograd.grad(clipped_objective, theta)[0]
g_composition = torch.tensor([1.0, -1.0])
raw_cos = F.cosine_similarity(g_raw[None], g_composition[None]).item()
clipped_cos = F.cosine_similarity(g_clipped[None], g_composition[None]).item()
print('Raw gradient:', g_raw.tolist())
print('Clipped gradient:', g_clipped.tolist())
print('Raw cosine:', round(raw_cos, 3))
print('Clipped cosine:', round(clipped_cos, 3))

## 5. Diagnostic tests — student exercise

Add an assertion that ensures (a) raw cosine is positive, (b) clipped cosine is negative. Then test the gradient vectors with `torch.testing.assert_close`. The reference expectations for this toy setup are `g_raw=[2.6,1]` and `g_clipped=[0,1]`.

In [ ]:
# TODO 3: replace `pass` with your assertions.
# Helpful examples: assert raw_cos > 0 ; torch.testing.assert_close(g_raw, torch.tensor([...]))
pass

## 6. Turn observations into a simple APT report

This reports a verified **toy clipping sign-flip mechanism**, not a claim that clipping caused a production held-out regression.

In [ ]:
report = {
    'experiment_id': 'toy_grpo_clipping_01',
    'evidence_scope': 'controlled_toy_surrogate',
    'ratios': [float(v) for v in ratios.detach()],
    'clip_epsilon': epsilon,
    'raw_gradient_cosine': raw_cos,
    'clipped_gradient_cosine': clipped_cos,
    'sign_flip': bool(raw_cos > 0 and clipped_cos < 0),
    'diagnosis': 'clipping_changes_gradient_direction_in_this_toy_example',
}
import json
from pathlib import Path
Path('apt_lab01_report.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

## 7. Investigate — don't just copy outputs

1. Change the first `initial_ratios` value from `1.3` to `1.1`. What happens to the clipped gradient?
2. Change its advantage to `-1` and reason about which side of the clipping interval suppresses negative-advantage gradients.
3. Explain why a sign flip **does not prove** that clipping caused a real held-out accuracy regression.

**Submit to your tutor:** your two TODO lines, the assertion test, outputs, and brief explanation. The next notebook will preserve the report shape while moving to a real TRL `SFTTrainer` experiment with label-mask diagnostics.

## 5. Plot the clipped objective for positive and negative advantages

The GRPO/PPO-style sample surrogate is $\min(\rho A,\,\mathrm{clip}(\rho,1-\epsilon,1+\epsilon)A)$. The **minimum is taken *after* multiplying by the signed advantage**. For $A>0$, the objective stops increasing past the upper clip edge; for $A<0$, it stops improving as the ratio goes below the lower clip edge. This is ratio clipping, **not** global gradient-norm clipping.

**Predict:** Where will the flat region be for $A=+1$ versus $A=-1$?

In [ ]:
r = np.linspace(0.4, 1.6, 401)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for ax, a in zip(axes, [1.0, -1.0]):
    y_raw = r * a
    y_clip = np.clip(r, 1-epsilon, 1+epsilon) * a
    y_obj = np.minimum(y_raw, y_clip)
    ax.plot(r, y_raw, '--', label='Unclipped ρA')
    ax.plot(r, y_obj, label='GRPO clipped surrogate')
    ax.axvspan(1-epsilon, 1+epsilon, alpha=.09, color='gray', label='Clip interval')
    ax.axvline(1-epsilon, color='gray', linestyle=':')
    ax.axvline(1+epsilon, color='gray', linestyle=':')
    ax.set(title=f'Advantage A = {a:+.0f}', xlabel='Importance ratio ρ', ylabel='Surrogate contribution')
    ax.legend(fontsize=8)
    ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

## 6. Plot the actual derivative across the clipping interval

The **slope** of the surrogate with respect to ratio $\rho$ is not the same as the parameter gradient $\nabla_\theta J$; the chain rule multiplies it by $\nabla_\theta\rho$. But plotting the slope shows exactly which terms are suppressed. We avoid the nondifferentiable clipping boundaries in the sampled grid.

In [ ]:
ratio_samples = torch.tensor(np.linspace(0.405, 1.595, 240), dtype=torch.float64, requires_grad=True)
fig, ax = plt.subplots(figsize=(10, 4))
for a in [1.0, -1.0]:
    u = ratio_samples * a
    q = ratio_samples.clamp(1-epsilon, 1+epsilon) * a
    objective = torch.minimum(u, q).sum()
    derivative = torch.autograd.grad(objective, ratio_samples, retain_graph=True)[0]
    ax.plot(ratio_samples.detach().numpy(), derivative.detach().numpy(), label=f'A = {a:+.0f}')
for x in [1-epsilon, 1+epsilon]:
    ax.axvline(x, color='gray', linestyle=':')
ax.set(xlabel='Ratio ρ', ylabel='d clipped surrogate / dρ', title='Autograd verifies where ratio gradients are suppressed')
ax.legend()
ax.grid(alpha=.2)
plt.show()

## 7. Visualize why gradient cosine changes sign

The blue arrow is the **raw atomic** gradient, the orange arrow is the **clipped atomic** gradient, and the green arrow is the fixed **composition diagnostic** gradient. Cosine similarity concerns direction, not magnitude. Positive raw cosine with negative clipped cosine is possible because one contribution is removed.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
vectors = [
    ('Raw atomic', g_raw.detach().numpy(), 'tab:blue'),
    ('Clipped atomic', g_clipped.detach().numpy(), 'tab:orange'),
    ('Composition diagnostic', g_composition.numpy(), 'tab:green'),
]
for label, vec, color in vectors:
    ax.arrow(0, 0, vec[0], vec[1], length_includes_head=True, head_width=.09, color=color, linewidth=2, label=label)
ax.axhline(0, color='gray', linewidth=.6)
ax.axvline(0, color='gray', linewidth=.6)
ax.set(xlim=(-1.5, 3.2), ylim=(-1.6, 2.2), xlabel='θ₁ direction', ylabel='θ₂ direction',
       title=f'Raw cosine = {raw_cos:.3f}; clipped cosine = {clipped_cos:.3f}')
ax.set_aspect('equal', adjustable='box')
ax.grid(alpha=.2)
ax.legend()
plt.show()

## 8. Sensitivity study: vary the clipping threshold

We preserve the same ratios, advantages, and feature vectors, changing only $\epsilon$. At $\rho_0=1.3$, the first positive-advantage sample is suppressed whenever $1.3>1+\epsilon$. **Predict** the threshold at which its contribution becomes active again. At the exact equality boundary, autograd selects a subgradient; avoid treating that one point as a stable derivative.

In [ ]:
eps_values = [0.05, 0.10, 0.20, 0.25, 0.31, 0.40]
clipped_cosines = []
first_sample_grads = []
for eps in eps_values:
    t = torch.zeros(2, requires_grad=True)
    rr = torch.exp(torch.log(initial_ratios) + features @ t)
    per_sample = torch.minimum(rr * advantages, rr.clamp(1-eps, 1+eps) * advantages)
    grads0 = torch.autograd.grad(per_sample[0], t, retain_graph=True)[0]
    total_grad = torch.autograd.grad(per_sample.sum(), t)[0]
    first_sample_grads.append(grads0[0].item())
    clipped_cosines.append(F.cosine_similarity(total_grad[None], g_composition[None]).item())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(eps_values, first_sample_grads, marker='o')
axes[0].set(title='Sample 0: ∂ objective / ∂θ₁', xlabel='Clip epsilon', ylabel='Gradient contribution')
axes[1].plot(eps_values, clipped_cosines, marker='o')
axes[1].axhline(0, linestyle=':', color='gray')
axes[1].set(title='Gradient alignment changes with epsilon', xlabel='Clip epsilon', ylabel='Cosine with composition gradient')
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout()
plt.show()
print('epsilon, sample0-gradient, cosine')
for vals in zip(eps_values, first_sample_grads, clipped_cosines):
    print(f'{vals[0]:.2f}  {vals[1]:.3f}  {vals[2]:+.3f}')

## 9. Interpret and validate

Before looking at the reference solution, answer these research questions:

1. At what **upper ratio** does a positive-advantage sample stop contributing a gradient? What changes if its advantage is negative?
2. Which of our plots shows $dJ/d\rho$ and which plots $\nabla_\theta J$? Why are these not interchangeable?
3. If clipping turns raw cosine positive to clipped cosine negative, what is **established** and what remains **unproven** about real held-out coding accuracy?
4. What additional evidence would separate clipping effects from Adam and KL effects in a real TRL trainer?

**Validation gate:** finish TODOs, verify the sign-flip assertions, and confirm the plot of $dJ/d\rho$ has the correct plateau branches. Then re-run after setting the first ratio to `1.1`, explaining why sample 0 now contributes. Finally restore `1.3` so your report matches the reference experiment.

**Artifacts:** `apt_lab01_report.json` plus your code, plot interpretations, and any parameter-sweep observations. The next real trainer lab uses TRL `SFTTrainer` with supervised-token masking diagnostics.

## Reference solution (open only after attempting)

For TODO 1–2:

```python
unclipped = ratios * advantages
clipped = torch.clamp(ratios, 1 - epsilon, 1 + epsilon) * advantages
```

For TODO 3:

```python
assert raw_cos > 0
assert clipped_cos < 0
torch.testing.assert_close(g_raw, torch.tensor([2.6, 1.0]), atol=1e-5, rtol=1e-5)
torch.testing.assert_close(g_clipped, torch.tensor([0.0, 1.0]))
```

Run outputs will be generated only when you execute the notebook.